# 04. Production exports, dashboard and updates
Start with notebook 01. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## Connect Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Get main from GitHub into temporary Colab storage
Code is freshly cloned from main into temporary Colab storage. Drive is used only for persistent artifacts. The old Drive checkout is not pulled or changed.


In [ ]:
%%bash
set -e
CODE_DIR=$(mktemp -d /content/foodprice_watch_code_XXXXXX)
git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git "$CODE_DIR"
printf '%s' "$CODE_DIR" > /content/foodprice_watch_code_path


## Initialize project runtime
Persistent artifacts are saved under the Drive project folder.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, csv, shutil
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
CODE_ROOT = Path('/content/foodprice_watch_code_path').read_text().strip()
CODE_ROOT = Path(CODE_ROOT)
os.chdir(CODE_ROOT)
os.environ['PYTHONPATH'] = str(CODE_ROOT / 'src')
os.environ['FPW_ARTIFACT_ROOT'] = str(ROOT)
assert (CODE_ROOT / 'src/fpw/runner.py').is_file()
ROOT.mkdir(parents=True, exist_ok=True)
# Seed only missing supporting files. Existing Drive results/state stay intact.
for directory in ['state', 'reports', 'config']:
    for original in (CODE_ROOT / directory).rglob('*'):
        if original.is_file():
            destination = ROOT / original.relative_to(CODE_ROOT)
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(original, destination)
for name in ['DECISIONS.md', 'PROGRESS.md', 'DEVIATIONS.md']:
    if not (ROOT / name).exists():
        shutil.copy2(CODE_ROOT / name, ROOT / name)
try:
    json.loads((ROOT / 'state/stages.json').read_text())
except ValueError as error:
    raise RuntimeError('Drive stage ledger is invalid, possibly due to old merge conflicts. Preserve it and ask for recovery; do not run stages.') from error
print('Code:', CODE_ROOT, '\nArtifacts:', ROOT)
sys.path.insert(0, str(CODE_ROOT / 'src'))
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))


## 3. Install runtime packages


In [ ]:
%pip install -q -r requirements-colab.txt


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))


In [ ]:
!python -m fpw.check


In [ ]:
def summarize_unit(sid):
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=stage.get('git_commit')), indent=2))


## S30: Production training and exports
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S31: Local dashboard instructions
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S32: Production update workflow
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


Testing requires committed pre-registration and existing checkpoint enforcement. Do not rerun earlier sealed tests. Keep models, exports and data out of GitHub.
